## tl;dr

严格 MBO 恢复后的 2026-07-09、07-21、08-07 三个完整日，日内 70/30 准确率为 59.24%–60.61%；严格跨日准确率为 58.48%–63.35%。但 DeepLOB 标准方向标签没有转化成可交易收益：下一快照进场、持有20档且计入点差后，四个主要跨日测试平均亏损 4.83–8.24bp/笔。将标签改成扣除点差和5.54bp成本后的 PnL 动作，并测试20/50/100档，也没有出现稳定正收益。当前结论是：模型有方向分类信息，但还不是可赚钱策略。

## Context & Methods

目标是判断25个严格时间因果特征的逻辑回归能否从分类器转化为07709的可执行策略。主检验使用逐日70/30和跨日 walk-forward；MBO 与 MBP 恢复口径不混合训练。

### Key Assumptions

- 信号在 t 形成，t+1 以卖一买入或买一卖出，t+1+h 以对手价退出。
- 禁止重叠持仓；点差已包含在成交价中。
- 官方固定征费和结算费往返约2.54bp；5.54bp情景再加入3bp往返券商佣金。
- 不计排队、冲击、延迟、最低佣金和融券费，因此结果仍偏乐观。
- ‘因果’只表示特征在时间上不使用未来信息，不是因果推断。

## Data

In [ ]:
from pathlib import Path
import json
import pandas as pd

result_path = Path('output/multiday_strategy_results.json')
trade_path = Path('output/multiday_strategy_trades.csv')
results = json.loads(result_path.read_text(encoding='utf-8'))
trades = pd.read_csv(trade_path)
print(f"Results: {result_path.resolve()}")
print(f"Trades: {len(trades):,}")

### Data-quality status

In [ ]:
quality = pd.DataFrame(results['data_quality'])
quality[['date', 'family', 'snapshots', 'first_send_time', 'last_send_time', 'final_state_tainted', 'order_errors']]

## Results

In [ ]:
classification_rows = []
for group_name in ('per_day_70_30_mbo', 'walk_forward_mbo_recent', 'walk_forward_mbp_adjacent'):
    for experiment in results['experiments'][group_name]:
        metric = experiment['classification']
        classification_rows.append({
            'group': group_name, 'name': experiment['name'],
            'train_dates': ','.join(experiment['train_dates']),
            'test_date': experiment['test_date'], 'samples': metric['samples'],
            'accuracy': metric['accuracy'], 'balanced_accuracy': metric['balanced_accuracy'],
            'macro_f1': metric['macro_f1']})
classification = pd.DataFrame(classification_rows)
classification.style.format({'accuracy': '{:.2%}', 'balanced_accuracy': '{:.2%}', 'macro_f1': '{:.2%}'})

### Standard-label strategy, strict walk-forward

In [ ]:
strategy_rows = []
for group_name in ('walk_forward_mbo_recent', 'walk_forward_mbp_adjacent'):
    for experiment in results['experiments'][group_name]:
        strategy = experiment['strategies']['long_short_p0.0']
        net = strategy['cost_scenarios']['5.54']
        strategy_rows.append({
            'name': experiment['name'], 'trades': strategy['trades'],
            'gross_mean_bps_after_spread': strategy['gross_mean_bps_after_spread'],
            'net_mean_bps_at_5.54': net['mean_net_bps'], 'win_rate_at_5.54': net['win_rate']})
strategy_table = pd.DataFrame(strategy_rows)
strategy_table.style.format({'gross_mean_bps_after_spread': '{:.2f}', 'net_mean_bps_at_5.54': '{:.2f}', 'win_rate_at_5.54': '{:.2%}'})

### PnL-aligned label and horizon sensitivity

In [ ]:
pnl_rows = []
for experiment in results['experiments']['pnl_aligned_walkforward']:
    strategy = experiment['strategies']['long_short_p0.0']
    pnl_rows.append({
        'name': experiment['name'], 'horizon': experiment['holding_horizon'],
        'class_weight': str(experiment['class_weight']), 'trades': strategy['trades'],
        'gross_mean_bps_after_spread': strategy['gross_mean_bps_after_spread'],
        'net_mean_bps_at_5.54': strategy['cost_scenarios']['5.54']['mean_net_bps']})
pnl_table = pd.DataFrame(pnl_rows)
pnl_table.sort_values('gross_mean_bps_after_spread', ascending=False, na_position='last').style.format({'gross_mean_bps_after_spread': '{:.2f}', 'net_mean_bps_at_5.54': '{:.2f}'})

## Takeaways

1. 多日和跨日结果证明分类信号并非完全随机，但标准标签奖励的是均价方向，不是可兑现 PnL。
2. 买一卖一点差已经大于模型可兑现的短周期边际；加入官方固定费用只会进一步恶化。
3. PnL 对齐后，自然类别权重模型几乎总是空仓；强行类别平衡会增加交易，但平均收益仍为负。
4. 因此当前版本不应实盘。若继续研究，优先方向不是继续堆分类准确率，而是使用更细采样/事件级执行标签、成交概率和排队模型，并取得连续数周数据做完全未触碰的最终测试。